# MicroDAW

## 1. Configuração

Importações e parâmetros globais usados em todo o notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display

from microsynth import (
    gera_tempo, midi2freq, freq2midi,
    adsr, sintetiza, fm, am, fm_time_warp, filtro,
    Instrumento,
    sintetiza_percussao, InstrumentoPercussivo
)

from microplayer import ArqMIDI, Player

from microview import (
    plot_sinais, plot_envelope, plot_espectrograma,
    plot_fft, plot_envelope_sinal, plot_energia,
    plot_pianoroll, plot_pianoroll_multivoz
)

sr = 44100
midi_dir  = '/midi/'
wav_dir = '/wav/'

## 2. Evolução E - Sintetizador Percussivo

### 2.1 Sons percussivos individuais

In [ ]:
# Gera e exibe cada tipo percussivo
percussoes = [
    ('bumbo', 45.0,  0.7),   # f_tonal mais grave (45 Hz)
    ('caixa', 200.0, 0.4),
    ('hihat', None,  0.2),
    ('tabla', 120.0, 0.5),
]

for tipo, f_ref, dur in percussoes:

    kwargs = dict(tipo=tipo, dur=dur, amp=0.9, sr=sr)

    if f_ref is not None:
        kwargs['f_tonal'] = f_ref

    y = sintetiza_percussao(**kwargs)

    print(f"{tipo.upper():8s} ({dur}s) →", end="  ")

    display(Audio(y, rate=sr))


### 2.2 Comparação de formas de onda e envelopes percussivos

In [ ]:
dur_comp = 0.7
sinais_perc = []
labels_perc = []

for tipo, f_ref in [('bumbo', 45.0), ('caixa', 200.0), ('hihat', None), ('tabla', 120.0)]:
    kwargs = dict(tipo=tipo, dur=dur_comp, amp=0.9, sr=sr)
    if f_ref:
        kwargs['f_tonal'] = f_ref
    sinais_perc.append(sintetiza_percussao(**kwargs))
    labels_perc.append(tipo)

plot_sinais(sinais_perc, labels=labels_perc, sr=sr,
            titulo='Comparação de formas de onda — sons percussivos')


## 3. Instrumentos Próprios

### 3.1 Sino

In [ ]:
sino = Instrumento(
    nome='Sino',
    forma='fm',
    n_harm=1,
    adsr_params=(
        0.001,                   # ataque instantâneo
        0.4,                     # decaimento suave
        0.3,                     # sustain baixo
        1.2                      # release longo — ressoa
    ),
    fase=0.0,
    unidade_fase='graus',
    am_params=None,
    fm_params={
        'f_c': 440.0,
        'f_m': 1.4,
        'I': 5.0,
        'tipo_fm': 'mult'
    },
    filtro_params={
        'cutoff_low':  200.0,
        'cutoff_high': 8000.0,
        'metodo': 'filtfilt'
    }
)

# Gera e exibe nota C5 (MIDI 72)
f_sino = midi2freq(72)
t_sino, y_sino = sino.gerar_nota(f=f_sino, dur=2.0, sr=sr, amp=0.9, retorna_t=True)

print(f"Sino — nota C5 ({f_sino:.1f} Hz) — {len(y_sino)/sr:.1f}s")
display(Audio(y_sino, rate=sr))

### 3.2 Tabla

In [ ]:
tabla = InstrumentoPercussivo(
    nome='Tabla (Bayan)',
    tipo_perc='tabla',
    f_tonal=100.0,    # frequência de referência — range real: 80–180 Hz
    dur_padrao=0.45   # acomoda o decaimento natural da membrana
)

# Demonstração com nota única
f_tabla = midi2freq(60)   # C4
y_tabla = tabla.gerar_nota(f=f_tabla, dur=0.1, amp=0.9, sr=sr)

print(f"Tabla — C4 ({f_tabla:.1f} Hz) — {len(y_tabla)/sr:.3f}s (dur_padrao usado)")
display(Audio(y_tabla, rate=sr))


### 3.3 Piano

In [ ]:
piano = Instrumento(
    nome=f'Piano',
    forma='dente',
    n_harm=12,
    adsr_params=(0.005, 0.30, 0.45, 0.40),
    fase=0.0,
    unidade_fase='graus',
    am_params=None,
    fm_params=None,
    filtro_params={
        'cutoff_low': None,
        'cutoff_high': 4500.0,
        'metodo': 'filtfilt'
    }
)

# Demonstração: nota A4 com duração mais longa para ouvir o decaimento
f_piano = midi2freq(69)   # A4 = 440 Hz
y_piano = piano.gerar_nota(f=f_piano, dur=1.5, amp=0.9, sr=sr)

print(f"Piano — A4 ({f_piano:.1f} Hz) — {len(y_piano)/sr:.2f}s")
display(Audio(y_piano, rate=sr))


### 3.3 Comparação espectral dos instrumentos

In [ ]:
# Gera os sinais de cada instrumento para a mesma nota (A4)
f_ref  = midi2freq(69)   # A4 = 440 Hz
dur_ref = 1.5

y_s = sino.gerar_nota(f=f_ref, dur=dur_ref, sr=sr, amp=0.9)
y_t = tabla.gerar_nota(f=f_ref, dur=dur_ref, sr=sr, amp=0.85)
y_v = piano.gerar_nota(f=f_ref, dur=dur_ref, sr=sr, amp=0.85)

print('Sino:')
display(Audio(y_s, rate=sr))
print('Tabla:')
display(Audio(y_t, rate=sr))
print('Piano:')
display(Audio(y_v, rate=sr))


In [ ]:
plot_espectrograma(y_s, sr)

In [ ]:
plot_espectrograma(y_t, sr)

In [ ]:
plot_espectrograma(y_v, sr)

## 4. Evolução C — Envelope Dependente de Velocity



In [ ]:
# Demo da Evolução C: mesma nota, três velocities diferentes

print('Tabla — Evolução C (velocity afeta frequência e amplitude):')
for amp_v, rotulo in [(0.3, 'suave  (amp=0.3)'), (0.7, 'média  (amp=0.7)'), (1.0, 'forte  (amp=1.0)')]:
    y_ev = tabla.gerar_nota(f=midi2freq(60), dur=0.1, amp=amp_v, sr=sr)
    print(f'  {rotulo}')
    display(Audio(y_ev, rate=sr))


## 5.MIDI


### 5.1 Carregar arquivo MIDI

In [ ]:
musica = 'voando_pro_para.mid'
arq_midi = midi_dir + musica

midi = ArqMIDI(arq_midi)

print(f"Arquivo: {arq_midi}")
print(f"BPM: {midi.bpm:.1f}")
print(f"Duração: {midi.duracao:.2f} s")

print()

print("Partes/vozes encontradas:")
print(f"  {'#':>3}  {'Nome':<30}  {'Percussiva':>10}  {'Eventos':>8}")
print("  " + "-"*56)

for i, (nome, perc, n_ev) in enumerate(midi.getPartList()):
    print(f"  {i:>3}  {nome:<30}  {'Sim' if perc else 'Não':>10}  {n_ev:>8}")

### 5.2 Visualizar piano roll

#### Uma voz

In [ ]:
plot_pianoroll(midi.getPart(0), voz=0)

#### Multi-voz

In [ ]:
# Piano roll multi-voz (primeiras vozes não-percussivas)
partes = midi.getPartList()
vozes_tonais = [i for i, (_, perc, n) in enumerate(partes) if not perc and n > 0]

lista_ev = [midi.getPart(i) for i in vozes_tonais[:4]]  # até 4 vozes
nomes_ev = [partes[i][0] for i in vozes_tonais[:4]]

plot_pianoroll_multivoz(lista_ev, nomes_vozes=nomes_ev)


### 5.3 Configurar instrumentos e renderizar

In [ ]:
partes_info = midi.getPartList()
n_vozes = len(partes_info)

print(partes_info)

lista_instrumentos = []

for i, (nome, percussiva, n_eventos) in enumerate(partes_info):
    if percussiva:
        instr = tabla
    else:
        instr = piano
    lista_instrumentos.append(instr)

In [ ]:
pl = Player()
pl.setArq(arq_midi, sr=sr)
pl.setInstrumentos(lista_instrumentos)

audio_stereo = pl.processa(canais=2, verbose=True)

print(f"\nRenderização concluída.")
print(f"Shape do áudio: {audio_stereo.shape}")
print(f"Duração: {audio_stereo.shape[-1] / sr:.2f} s")


## 6. Reprodução e Exportação

O áudio gerado pode ser reproduzido diretamente no notebook ou salvo como `.wav`.


In [ ]:
display(Audio(audio_stereo, rate=sr))

In [ ]:
def salva_wav(caminho, y, sr=sr):

    if y.ndim == 2 and y.shape[0] == 2:
        y = y.T

    peak = np.max(np.abs(y))
    if peak > 1e-9:
        y = y / peak * 0.9

    wavfile.write(caminho, sr, (y * 32767).astype(np.int16))

    print(f"Salvo: {caminho}")

salva_wav(wav_dir + musica.replace('mid', 'wav'), audio_stereo)

## . Visualizações

Análise do sinal final usando as funções de `microview.py`.


### 7.1 Forma de onda final

In [ ]:
# Pega o canal mono para visualização
if audio_stereo.ndim == 2:
    audio_mono = audio_stereo[0]
else:
    audio_mono = audio_stereo

plot_sinais(
    [audio_mono],
    labels=['Waveform final'],
    sr=sr,
    titulo='Waveform final — música sintetizada'
)


### 7.2 Espectro de frequências (FFT)

In [ ]:
plot_fft(audio_mono, sr=sr, titulo='Espectro de frequências — música sintetizada')


### 7.3 Espectrograma

In [ ]:
plot_espectrograma(audio_mono, sr=sr)


### 7.4 Envelope de amplitude

In [ ]:
plot_envelope_sinal(audio_mono, sr=sr, janela=2048)


### 7.5 Energia por voz

In [ ]:
# Obtém os buffers separados por voz (canais=0)
buffers_vozes = pl.getWav(canais=0)

if buffers_vozes is not None and buffers_vozes.ndim == 2:
    plot_energia(buffers_vozes)
else:
    print("Buffers por voz não disponíveis — execute pl.processa() primeiro.")
